In [1]:
import re, unicodedata
import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error


In [2]:
import re
import pandas as pd

pos_map = [
    ("catcher", "C"), ("first base", "1B"), ("second base", "2B"),
    ("third base", "3B"), ("shortstop", "SS"),
    ("left field", "LF"), ("center field", "CF"), ("right field", "RF"),
    ("designated", "DH"), ("starting", "SP"), ("reliever", "RP"),
]

rows, pos = [], None
with open("../data/raw/mlbtr_2027_fa.txt", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        m = re.match(r"^(.+?)\s+\((\d{2})\)\s*(?:[–—-]\s*(.*))?$", line)
        if m:
            if pos:
                rows.append({
                    "player_name": m.group(1).strip(),
                    "position": pos,
                    "age_2027": int(m.group(2)),
                    "option_note": (m.group(3) or "").strip(),
                })
        elif len(line) < 40:
            for key, val in pos_map:
                if key in line.lower():
                    pos = val

raw = pd.DataFrame(rows)

# A player listed under several positions becomes one row
fa = (raw.groupby("player_name", sort=False)
         .agg(position=("position", "first"),
              all_positions=("position", lambda s: "/".join(dict.fromkeys(s))),
              age_2027=("age_2027", "first"),
              option_note=("option_note", lambda s: next((x for x in s if x), "")))
         .reset_index())

fa["player_type"] = fa["position"].apply(lambda p: "pitcher" if p in ["SP", "RP"] else "hitter")
fa["has_option"] = fa["option_note"] != ""

print(len(raw), "lines ->", len(fa), "unique players")
print(fa.groupby("position").size())
print(fa["has_option"].sum(), "players with an option or opt-out noted")
fa.to_csv("../data/processed/fa_2027_list.csv", index=False)
fa.head(10)

272 lines -> 257 unique players
position
1B    10
2B    17
3B    10
C     15
CF    11
DH     6
LF    13
RF    13
RP    90
SP    64
SS     8
dtype: int64
64 players with an option or opt-out noted


,player_name,position,all_positions,age_2027,option_note,player_type,has_option
0,Travis d’Arnaud,C,C,38,,hitter,False
1,Elias Diaz,C,C,36,,hitter,False
2,Mitch Garver,C,C,36,,hitter,False
3,Austin Hedges,C,C,34,,hitter,False
4,Jonah Heim,C,C,32,,hitter,False
5,Kyle Higashioka,C,C,37,$7MM mutual option with a $1MM buyout,hitter,True
6,Ryan Jeffers,C,C,30,,hitter,False
7,Carson Kelly,C,C,32,$7.5MM mutual option with a $1.5MM buyout,hitter,True
8,Luke Maile,C,C,36,,hitter,False
9,James McCann,C,C,37,,hitter,False


In [3]:
import re, time, unicodedata
import numpy as np
import pandas as pd
import requests

def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    s = s.replace("-", " ")
    s = re.sub(r"\b(jr|sr|ii|iii|iv)\b\.?", "", s)
    s = re.sub(r"[^a-z ]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return {"michael fiers": "mike fiers", "steven pearce": "steve pearce"}.get(s, s)

fa = pd.read_csv("../data/processed/fa_2027_list.csv")
bh = pd.read_csv("../data/processed/bwar_hitters.csv")
bp = pd.read_csv("../data/processed/bwar_pitchers.csv")
sh = pd.read_csv("../data/processed/savant_hitters.csv")
sp = pd.read_csv("../data/processed/savant_pitchers.csv")

# Only the 2024-2026 seasons matter for the 2027 class
bh = bh[bh["year_ID"] >= 2024].copy()
bp = bp[bp["year_ID"] >= 2024].copy()
bh["key"] = bh["name"].apply(norm)
bp["key"] = bp["name"].apply(norm)
fa["key"] = fa["player_name"].apply(norm)

# 1. Candidate MLB IDs for each name
cands = {}
for t, df in [("hitter", bh), ("pitcher", bp)]:
    g = df.groupby("key")["mlb_ID"].unique()
    cands[t] = {k: [int(i) for i in v] for k, v in g.items()}
fa["cand_ids"] = [cands[t].get(k, []) for t, k in zip(fa["player_type"], fa["key"])]

# 2. Birth dates from the MLB Stats API
all_ids = sorted({i for c in fa["cand_ids"] for i in c})
births = {}
for i in range(0, len(all_ids), 100):
    batch = all_ids[i:i + 100]
    r = requests.get("https://statsapi.mlb.com/api/v1/people",
                     params={"personIds": ",".join(map(str, batch))}, timeout=30)
    r.raise_for_status()
    for p in r.json().get("people", []):
        if "birthDate" in p:
            births[p["id"]] = p["birthDate"]
    time.sleep(0.5)

# 3. If a name has several candidates, use the age from the MLBTR list to choose
def pick(row):
    c = row["cand_ids"]
    if len(c) == 1:
        return c[0]
    ok = []
    for i in c:
        if i in births:
            by = int(births[i][:4])
            if ((2027 - by) - row["age_2027"]) in (0, 1):
                ok.append(i)
    return ok[0] if len(ok) == 1 else np.nan

fa["mlb_ID"] = fa.apply(pick, axis=1)
fa["birth_date"] = pd.to_datetime(fa["mlb_ID"].map(births))
fa["age_at_signing"] = ((pd.Timestamp("2026-12-01") - fa["birth_date"]).dt.days / 365.25).round(1)

# 4. Weighted WAR over 2026, 2025, 2024 (same 5-4-3 weights as the training data)
war = {}
for t, df in [("hitter", bh), ("pitcher", bp)]:
    for mid, yr, w in zip(df["mlb_ID"], df["year_ID"], df["WAR"]):
        war[(t, int(mid), int(yr))] = w

for k, col in enumerate(["war_1", "war_2", "war_3"]):
    fa[col] = [war.get((t, int(i), 2026 - k), 0.0) if pd.notna(i) else np.nan
               for t, i in zip(fa["player_type"], fa["mlb_ID"])]
fa["war_w"] = (5 * fa["war_1"] + 4 * fa["war_2"] + 3 * fa["war_3"]) / 12

# 5. Savant percentiles, weighted the same way
feats = {
    "hitter": ["xwoba", "brl_percent", "exit_velocity", "chase_percent",
               "whiff_percent", "bb_percent", "sprint_speed"],
    "pitcher": ["xera", "fb_velocity", "k_percent", "bb_percent",
                "whiff_percent", "chase_percent", "hard_hit_percent"],
}
sav = {"hitter": sh, "pitcher": sp}

def savant_2027(df, t):
    s, f = sav[t], feats[t]
    lookup = dict(zip(zip(s["player_id"].astype(int), s["year"].astype(int)),
                      s[f].values.astype(float)))
    out = []
    for mid in df["mlb_ID"]:
        if pd.isna(mid):
            out.append([np.nan] * len(f))
            continue
        num, den = np.zeros(len(f)), np.zeros(len(f))
        for k, w in enumerate([5, 4, 3]):
            arr = lookup.get((int(mid), 2026 - k))
            if arr is None:
                continue
            ok = ~np.isnan(arr)
            num[ok] += w * arr[ok]
            den[ok] += w
        out.append(np.where(den > 0, num / np.where(den > 0, den, 1), np.nan))
    return pd.DataFrame(out, columns=f, index=df.index)

sv = pd.concat([savant_2027(fa[fa["player_type"] == t], t) for t in ["hitter", "pitcher"]])
fa = fa.join(sv)

# 6. Model inputs that match the training columns
fa["age_sq"] = (fa["age_at_signing"] - 31) ** 2
fa["year"] = 2027 - 2015
fa["qualifying_offer"] = 0   # placeholder until offers are made after the World Series

fa.drop(columns=["cand_ids"]).to_csv("../data/processed/fa_2027_features.csv", index=False)

# Summary
has_sav = np.where(fa["player_type"] == "hitter", fa["xwoba"].notna(), fa["xera"].notna())
print("Players:", len(fa))
print("Matched to an MLB ID:", fa["mlb_ID"].notna().sum())
print("With Savant data:", int(has_sav.sum()))
print("\nNo ID (no 2024-2026 MLB season, or an ambiguous name):")
print(fa[fa["mlb_ID"].isna()][["player_name", "position", "age_2027"]].to_string(index=False))
print("\nTop 10 by weighted WAR:")
fa.nlargest(10, "war_w")[["player_name", "position", "age_at_signing", "war_3", "war_2", "war_1", "war_w"]]

Players: 257
Matched to an MLB ID: 253
With Savant data: 212

No ID (no 2024-2026 MLB season, or an ambiguous name):
   player_name position  age_2027
Kiké Hernandez       3B        35
   Kyle Wright       SP        31
Trevor Stephan       RP        31
    Rowan Wick       RP        34

Top 10 by weighted WAR:


,player_name,position,age_at_signing,war_3,war_2,war_1,war_w
157,Tarik Skubal,SP,30.0,6.66,6.39,4.60,5.711667
61,Randy Arozarena,LF,31.8,1.71,3.87,5.70,4.092500
95,Seiya Suzuki,RF,32.3,3.53,2.58,5.12,3.875833
149,Drew Rasmussen,SP,31.3,0.60,4.40,4.66,3.558333
66,Ian Happ,LF,32.3,3.92,3.96,2.86,3.491667
33,Brandon Lowe,2B,32.4,2.35,1.81,5.35,3.420000
135,Nick Martinez,SP,36.3,3.98,2.19,3.91,3.354167
130,Michael King,SP,31.5,4.07,1.19,4.26,3.189167
152,Trevor Rogers,SP,29.0,0.59,5.28,2.23,2.836667
28,Jazz Chisholm Jr.,2B,28.8,2.92,4.12,1.69,2.807500


In [4]:
import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

feats = {
    "hitter": ["xwoba", "brl_percent", "exit_velocity", "chase_percent",
               "whiff_percent", "bb_percent", "sprint_speed"],
    "pitcher": ["xera", "fb_velocity", "k_percent", "bb_percent",
                "whiff_percent", "chase_percent", "hard_hit_percent"],
}
sav = {
    "hitter": pd.read_csv("../data/processed/savant_hitters.csv"),
    "pitcher": pd.read_csv("../data/processed/savant_pitchers.csv"),
}

# Training contracts ($5M+, Ohtani excluded, qualifying-offer acceptances removed)
d = pd.read_csv("../data/processed/contracts_model.csv")
d = d[d["age_at_signing"].notna() & (d["player_name"] != "Shohei Ohtani")].copy()
d = d[d["total_value"] >= 5_000_000].copy()
d["aav_m"] = d["aav"] / 1e6
d["age_sq"] = (d["age_at_signing"] - 31) ** 2
d["year"] = d["signing_year"] - 2015

# 2027 free agents
fa = pd.read_csv("../data/processed/fa_2027_features.csv")

def savant_avg(df, t, year_col):
    s, f = sav[t], feats[t]
    lookup = dict(zip(zip(s["player_id"].astype(int), s["year"].astype(int)),
                      s[f].values.astype(float)))
    out = []
    for mid, ly in zip(df["mlb_ID"], df[year_col]):
        if pd.isna(mid):
            out.append([np.nan] * len(f))
            continue
        num, den = np.zeros(len(f)), np.zeros(len(f))
        for k, w in enumerate([5, 4, 3]):
            arr = lookup.get((int(mid), int(ly) - k))
            if arr is None:
                continue
            ok = ~np.isnan(arr)
            num[ok] += w * arr[ok]
            den[ok] += w
        out.append(np.where(den > 0, num / np.where(den > 0, den, 1), np.nan))
    return pd.DataFrame(out, columns=f, index=df.index)

def build(df, cols):
    X = pd.get_dummies(df[cols + ["position"]], columns=["position"], dtype=float)
    resid = []
    for yr in range(2021, 2027):          # rolling out-of-sample errors
        tr = (df["signing_year"] < yr).values
        te = (df["signing_year"] == yr).values
        if te.sum() == 0 or tr.sum() < 30:
            continue
        m = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30)))
        m.fit(X[tr], df.loc[tr, "aav_m"])
        resid.extend((df.loc[te, "aav_m"] - m.predict(X[te])).tolist())
    final = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30)))
    final.fit(X, df["aav_m"])             # final model uses ALL years
    lo, hi = np.percentile(resid, [10, 90])
    return final, list(X.columns), lo, hi

def predict(model, xcols, df, cols):
    X = pd.get_dummies(df[cols + ["position"]], columns=["position"], dtype=float)
    return model.predict(X.reindex(columns=xcols, fill_value=0.0))

base = ["age_at_signing", "age_sq", "qualifying_offer", "year"]
results, bands = [], {}

for t in ["hitter", "pitcher"]:
    f = feats[t]
    tr = d[d["player_type"] == t].copy()
    tr = pd.concat([tr, savant_avg(tr, t, "stat_year")], axis=1)

    colsA = base + ["war_w"]                       # Model A: WAR only
    mA, xA, loA, hiA = build(tr, colsA)

    trB = tr[tr[f[0]].notna()].copy()              # Model B: WAR + Savant
    trB[f] = trB[f].fillna(50)
    colsB = base + ["war_w"] + f
    mB, xB, loB, hiB = build(trB, colsB)
    bands[f"{t} WAR only"] = (loA, hiA)
    bands[f"{t} WAR + Savant"] = (loB, hiB)

    te = fa[(fa["player_type"] == t) & fa["mlb_ID"].notna()
            & fa["age_at_signing"].notna()].copy()
    has_sav = te[f[0]].notna().values
    te[f] = te[f].fillna(50)

    pA = predict(mA, xA, te, colsA)
    pB = predict(mB, xB, te, colsB)
    te["pred_aav_m"] = np.where(has_sav, pB, pA)
    te["low_m"] = (np.where(has_sav, loB, loA) + te["pred_aav_m"]).clip(lower=0)
    te["high_m"] = np.where(has_sav, hiB, hiA) + te["pred_aav_m"]
    te["model"] = np.where(has_sav, "WAR + Savant", "WAR only")
    results.append(te)

out = pd.concat(results)
out["below_5m_floor"] = out["pred_aav_m"] < 5
out["note"] = np.where(out["player_name"] == "Tarik Skubal",
                       "outlier: model likely far too low", "")

show = ["player_name", "position", "age_at_signing", "war_w", "pred_aav_m",
        "low_m", "high_m", "model", "option_note", "below_5m_floor", "note"]
out = out.sort_values("pred_aav_m", ascending=False)[show].round(1)
out.to_csv("../outputs/fa_2027_predictions.csv", index=False)

print("Predicted:", len(out), "of", len(fa), "players")
print(out["model"].value_counts().to_string())
print("\nRange used (past errors in $M, 10th to 90th percentile):")
for k, (lo, hi) in bands.items():
    print(f"  {k}: {lo:+.1f} to {hi:+.1f}")
print("\nUnder the $5M floor:", int(out["below_5m_floor"].sum()), "players")
out.head(15)

Predicted: 253 of 257 players
model
WAR + Savant    212
WAR only         41

Range used (past errors in $M, 10th to 90th percentile):
  hitter WAR only: -4.3 to +7.8
  hitter WAR + Savant: -4.1 to +6.7
  pitcher WAR only: -4.4 to +6.1
  pitcher WAR + Savant: -4.2 to +6.1

Under the $5M floor: 49 players


,player_name,position,age_at_signing,war_w,pred_aav_m,low_m,high_m,model,option_note,below_5m_floor,note
157,Tarik Skubal,SP,30.0,5.7,32.2,28.1,38.3,WAR + Savant,NaN,False,outlier: model likely far too low
95,Seiya Suzuki,RF,32.3,3.9,24.0,19.9,30.7,WAR + Savant,NaN,False,
149,Drew Rasmussen,SP,31.3,3.6,22.9,18.7,29.0,WAR + Savant,$20MM club option with a $500K buyout,False,
61,Randy Arozarena,LF,31.8,4.1,21.3,17.2,28.0,WAR + Savant,NaN,False,
84,Ronald Acuña Jr.,RF,29.0,1.8,20.8,16.7,27.5,WAR + Savant,$17MM club option with a $10MM buyout,False,
143,Freddy Peralta,SP,30.5,2.8,19.3,15.2,25.4,WAR + Savant,NaN,False,
66,Ian Happ,LF,32.3,3.5,19.2,15.1,25.9,WAR + Savant,NaN,False,
130,Michael King,SP,31.5,3.2,19.1,14.9,25.1,WAR + Savant,"can opt out of remaining two years, $58MM for ...",False,
74,Trent Grisham,CF,30.1,1.5,19.0,14.9,25.7,WAR + Savant,NaN,False,
152,Trevor Rogers,SP,29.0,2.8,18.2,14.0,24.2,WAR + Savant,NaN,False,


In [5]:
import pandas as pd

out = pd.read_csv("../outputs/fa_2027_predictions.csv")
notes = {
    "Tarik Skubal": "outlier: model output is a floor. Closest precedent by age and weighted WAR is Gerrit Cole (2020): "
                    "9 years, $324M ($36M AAV); top pitcher AAVs excluding Ohtani are about $34M-$43M. "
                    "MLBTR expects over $400M",
    "Ronald Acuña Jr.": "former MVP; 222/412/469 PA in 2024-2026, and his 2023 season (8.4 WAR) is outside the model's window; "
                        "a $17MM club option makes reaching the market unlikely",
    "Kyle Freeland": "2027 player option did not vest (under 170 IP in 2026); treated as a free agent",
}
out["note"] = out["player_name"].map(notes).fillna(out["note"])
out.to_csv("../outputs/fa_2027_predictions.csv", index=False)
out[out["note"].notna()][["player_name", "pred_aav_m", "note"]]

,player_name,pred_aav_m,note
0,Tarik Skubal,32.2,outlier: model output is a floor. Closest prec...
4,Ronald Acuña Jr.,20.8,"former MVP; 222/412/469 PA in 2024-2026, and h..."
153,Kyle Freeland,7.1,2027 player option did not vest (under 170 IP ...


In [6]:
import re, unicodedata
import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    s = s.replace("-", " ")
    s = re.sub(r"\b(jr|sr|ii|iii|iv)\b\.?", "", s)
    s = re.sub(r"[^a-z ]", "", s)
    return re.sub(r"\s+", " ", s).strip()

d = pd.read_csv("../data/processed/contracts_model.csv")
d = d[d["age_at_signing"].notna() & (d["player_name"] != "Shohei Ohtani")]
d = d[d["total_value"] >= 5_000_000].copy()
d["aav_m"] = d["aav"] / 1e6
d["age_sq"] = (d["age_at_signing"] - 31) ** 2
d["year"] = d["signing_year"] - 2015
d["key"] = d["player_name"].apply(norm)
d["years_c"] = d["years"].clip(upper=8)

cols = ["war_w", "age_at_signing", "age_sq", "qualifying_offer", "year"]
X = pd.get_dummies(d[cols + ["position"]], columns=["position"], dtype=float)

rows = []
for yr in range(2021, 2027):
    tr = (d["signing_year"] < yr).values
    te = (d["signing_year"] == yr).values
    m = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30)))
    m.fit(X[tr], d.loc[tr, "years_c"])
    part = d[te].copy()
    part["pred_years"] = np.clip(m.predict(X[te]), 1, 8)
    part["naive_years"] = d.loc[tr, "years_c"].median()
    rows.append(part)

r = pd.concat(rows)
r["err"] = r["pred_years"] - r["years_c"]
r["naive_err"] = r["naive_years"] - r["years_c"]
r["size"] = pd.cut(r["aav_m"], [0, 10, 20, 100], labels=["under $10M", "$10-20M", "$20M+"])

print("Contract length, rolling test 2021-2026 (%d contracts)" % len(r))
print("  Model MAE: %.2f years | Naive (median) MAE: %.2f years" % (r["err"].abs().mean(), r["naive_err"].abs().mean()))
print("  Within 1 year of actual: model %.0f%% | naive %.0f%%" % (
    100 * (r["err"].abs() <= 1).mean(), 100 * (r["naive_err"].abs() <= 1).mean()))
print("\nBy actual contract size:")
print(r.groupby("size", observed=True).agg(
    n=("err", "size"), actual_years=("years_c", "mean"), predicted_years=("pred_years", "mean"),
    model_mae=("err", lambda s: s.abs().mean()), naive_mae=("naive_err", lambda s: s.abs().mean())).round(2))

print("\nYears for $20M+ contracts, by class:")
print(d[d["aav_m"] >= 20].groupby("signing_year")["years"].agg(["count", "mean"]).round(1).T.to_string())

# Head to head with MLBTR on the same players (2024-2026 classes)
b = pd.read_csv("../outputs/benchmark_mlbtr_vs_model.csv")
b = b.merge(r[["key", "signing_year", "pred_years"]].drop_duplicates(["key", "signing_year"]),
            on=["key", "signing_year"], how="left")
b = b[b["pred_years"].notna()].copy()
b["model_total_m"] = b["model_aav_m"] * b["pred_years"]

def head_to_head(g, label):
    return pd.Series({
        "n": len(g),
        "years MAE MLBTR": (g["mlbtr_years"] - g["actual_years"]).abs().mean(),
        "years MAE model": (g["pred_years"] - g["actual_years"]).abs().mean(),
        "total MAE MLBTR ($M)": (g["mlbtr_total_m"] - g["actual_total_m"]).abs().mean(),
        "total MAE model ($M)": (g["model_total_m"] - g["actual_total_m"]).abs().mean(),
        "total bias MLBTR": (g["mlbtr_total_m"] - g["actual_total_m"]).mean(),
        "total bias model": (g["model_total_m"] - g["actual_total_m"]).mean(),
    }, name=label)

print("\nHead to head with MLBTR (same players):")
print(pd.concat([head_to_head(b, "all scored"),
                 head_to_head(b[b["actual_aav_m"] >= 20], "$20M+ only")], axis=1).round(1).to_string())

Contract length, rolling test 2021-2026 (433 contracts)
  Model MAE: 0.81 years | Naive (median) MAE: 1.03 years
  Within 1 year of actual: model 71% | naive 87%

By actual contract size:
              n  actual_years  predicted_years  model_mae  naive_mae
size                                                                
under $10M  231          1.44             1.64       0.59       0.69
$10-20M     144          2.19             2.08       0.80       0.92
$20M+        58          4.55             3.70       1.76       2.69

Years for $20M+ contracts, by class:
signing_year  2015  2016  2017  2018  2019  2020  2021  2022  2023  2024  2025  2026
count          3.0   7.0   2.0   4.0   5.0   6.0   3.0  15.0  13.0   7.0  14.0  14.0
mean           5.7   5.9   3.5   4.2   6.2   6.0   4.7   4.7   5.8   3.0   4.6   3.8

Head to head with MLBTR (same players):
                      all scored  $20M+ only
n                          120.0        35.0
years MAE MLBTR              0.9         1.

In [7]:
import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

d = pd.read_csv("../data/processed/contracts_model.csv")
d = d[d["age_at_signing"].notna() & (d["player_name"] != "Shohei Ohtani")]
d = d[d["total_value"] >= 5_000_000].copy()
d["age_sq"] = (d["age_at_signing"] - 31) ** 2
d["year"] = d["signing_year"] - 2015
d["years_c"] = d["years"].clip(upper=8)

# Check of the star-deal split quoted earlier
star = d[(d["aav"] / 1e6) >= 20]
print(star.groupby(star["signing_year"] >= 2024)["years"].agg(["count", "mean"]).round(2)
      .rename(index={False: "2015-2023", True: "2024-2026"}))

cols = ["war_w", "age_at_signing", "age_sq", "qualifying_offer", "year"]
X = pd.get_dummies(d[cols + ["position"]], columns=["position"], dtype=float)
model = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30)))
model.fit(X, d["years_c"])                      # final model uses all classes

fa = pd.read_csv("../data/processed/fa_2027_features.csv")
fa = fa[fa["mlb_ID"].notna() & fa["age_at_signing"].notna()].copy()
Xf = pd.get_dummies(fa[cols + ["position"]], columns=["position"], dtype=float)
Xf = Xf.reindex(columns=X.columns, fill_value=0.0)
fa["pred_years_raw"] = np.clip(model.predict(Xf), 1, 8)

pred = pd.read_csv("../outputs/fa_2027_predictions.csv")
out = pred.merge(fa[["player_name", "pred_years_raw"]], on="player_name", how="left")
out["pred_years"] = out["pred_years_raw"].round().clip(lower=1)
out["years_low"] = (out["pred_years"] - 1).clip(lower=1)
out["years_high"] = out["pred_years"] + 1
out["pred_total_m"] = (out["pred_aav_m"] * out["pred_years"]).round(0)
out["lower_bound"] = out["pred_aav_m"] >= 20

keep = ["player_name", "position", "age_at_signing", "pred_aav_m", "pred_years",
        "years_low", "years_high", "pred_total_m", "lower_bound", "below_5m_floor",
        "option_note", "note"]
out = out.sort_values("pred_total_m", ascending=False)[keep]
out.to_csv("../outputs/fa_2027_contract_projections.csv", index=False)

print("\nPlayers:", len(out), "| missing years:", int(out["pred_years"].isna().sum()))
print("Predicted years:", out["pred_years"].value_counts().sort_index().to_dict())
out.head(15)

              count  mean
signing_year             
2015-2023        58  5.34
2024-2026        35  3.97

Players: 253 | missing years: 0
Predicted years: {1.0: 156, 2.0: 81, 3.0: 15, 5.0: 1}


,player_name,position,age_at_signing,pred_aav_m,pred_years,years_low,years_high,pred_total_m,lower_bound,below_5m_floor,option_note,note
0,Tarik Skubal,SP,30.0,32.2,5.0,4.0,6.0,161.0,True,False,NaN,outlier: model output is a floor. Closest prec...
1,Seiya Suzuki,RF,32.3,24.0,3.0,2.0,4.0,72.0,True,False,NaN,NaN
2,Drew Rasmussen,SP,31.3,22.9,3.0,2.0,4.0,69.0,True,False,$20MM club option with a $500K buyout,NaN
3,Randy Arozarena,LF,31.8,21.3,3.0,2.0,4.0,64.0,True,False,NaN,NaN
4,Ronald Acuña Jr.,RF,29.0,20.8,3.0,2.0,4.0,62.0,True,False,$17MM club option with a $10MM buyout,"former MVP; 222/412/469 PA in 2024-2026, and h..."
5,Freddy Peralta,SP,30.5,19.3,3.0,2.0,4.0,58.0,False,False,NaN,NaN
6,Ian Happ,LF,32.3,19.2,3.0,2.0,4.0,58.0,False,False,NaN,NaN
7,Michael King,SP,31.5,19.1,3.0,2.0,4.0,57.0,False,False,"can opt out of remaining two years, $58MM for ...",NaN
9,Trevor Rogers,SP,29.0,18.2,3.0,2.0,4.0,55.0,False,False,NaN,NaN
10,Ryan Jeffers,C,29.5,18.1,3.0,2.0,4.0,54.0,False,False,NaN,NaN


In [8]:
import re
import numpy as np
import pandas as pd

p = pd.read_csv("../outputs/fa_2027_contract_projections.csv")
o = p[p["option_note"].notna() & (p["option_note"].astype(str).str.strip() != "")].copy()

words = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5}

def money(txt):
    m = re.search(r"\$([\d.]+)\s*(MM|K)\b", txt)
    if not m:
        return np.nan
    v = float(m.group(1))
    return v if m.group(2) == "MM" else v / 1000

def parse_option(note):
    t = str(note)
    low = t.lower()
    r = {"option_type": "unknown", "amount_m": np.nan, "buyout_m": np.nan, "optout_years": np.nan}
    if "opt out" in low:
        r["option_type"] = "opt-out"
        m = re.search(r"remaining\s+(\w+)\s+years?", low)
        if m:
            r["optout_years"] = words.get(m.group(1), np.nan)
    elif "mutual option" in low:
        r["option_type"] = "mutual"
    elif "club option" in low:
        r["option_type"] = "club"
    elif "player option" in low:
        r["option_type"] = "player"
    r["amount_m"] = money(t)   # option salary, or money remaining for an opt-out
    b = re.search(r"\$([\d.]+)\s*(MM|K)\s*buyout", t)
    if b:
        v = float(b.group(1))
        r["buyout_m"] = v if b.group(2) == "MM" else v / 1000
    elif "no buyout" in low:
        r["buyout_m"] = 0.0
    r["needs_review"] = (bool(re.search(r"\bif\b|innings|vest", low))
                         or r["option_type"] == "unknown" or np.isnan(r["amount_m"]))
    return pd.Series(r)

o = pd.concat([o, o["option_note"].apply(parse_option)], axis=1)

print("Players with an option or opt-out:", len(o))
print(o["option_type"].value_counts().to_string())

print("\nNeeds a manual look:")
print(o[o["needs_review"]][["player_name", "option_type", "amount_m", "buyout_m", "option_note"]]
      .to_string(index=False))

cols = ["player_name", "option_type", "amount_m", "buyout_m", "optout_years",
        "pred_aav_m", "pred_years", "pred_total_m", "needs_review"]
o[cols].to_csv("../data/processed/fa_2027_options.csv", index=False)
o.sort_values("pred_total_m", ascending=False)[cols].head(20)

Players with an option or opt-out: 62
option_type
club       25
mutual     25
opt-out     8
player      4

Needs a manual look:
    player_name option_type  amount_m  buyout_m                                                                                                                           option_note
Aroldis Chapman      mutual      13.0       0.3 $13MM mutual option with a $300K buyout.  Guaranteed if Chapman pitches 40 innings in 2026 (met) and passes post-season physical exam
  Kyle Freeland      player      17.0       NaN                                                                         $17MM player option for 2027 with 170 innings pitched in 2026
   Gary Sanchez      mutual       NaN       NaN                                                                                                                         mutual option


,player_name,option_type,amount_m,buyout_m,optout_years,pred_aav_m,pred_years,pred_total_m,needs_review
2,Drew Rasmussen,club,20.00,0.5,NaN,22.9,3.0,69.0,False
4,Ronald Acuña Jr.,club,17.00,10.0,NaN,20.8,3.0,62.0,False
7,Michael King,opt-out,58.00,5.0,2.0,19.1,3.0,57.0,False
11,Andres Muñoz,club,8.00,NaN,NaN,15.2,3.0,46.0,False
14,Garrett Whitlock,club,8.25,1.0,NaN,13.8,3.0,41.0,False
16,Nick Pivetta,club,14.00,0.0,NaN,18.0,2.0,36.0,False
17,Nick Martinez,mutual,20.00,4.0,NaN,17.4,2.0,35.0,False
19,Emmanuel Clase,club,10.00,2.0,NaN,11.8,3.0,35.0,False
20,Sonny Gray,mutual,30.00,10.0,NaN,16.9,2.0,34.0,False
23,Aroldis Chapman,mutual,13.00,0.3,NaN,16.0,2.0,32.0,True


In [9]:
import numpy as np
import pandas as pd

o = pd.read_csv("../data/processed/fa_2027_options.csv")
p = pd.read_csv("../outputs/fa_2027_contract_projections.csv")
o = o.merge(p[["player_name", "position", "age_at_signing", "option_note"]],
            on="player_name", how="left")

MARGIN = 4.0   # about the model's average AAV miss, in $M

def band(diff):
    a = abs(diff)
    return "toss-up" if a <= MARGIN else ("lean" if a <= 2 * MARGIN else "strong")

def decide(r):
    t, amt, pred = r["option_type"], r["amount_m"], r["pred_aav_m"]
    buy = 0.0 if pd.isna(r["buyout_m"]) else r["buyout_m"]
    if pd.isna(amt):
        return np.nan, "Terms missing: needs a manual look", ""
    if t in ("club", "mutual"):
        net = amt - buy
        diff = pred - net
        reason = f"projected ${pred:.1f}M a year vs net option cost ${net:.1f}M"
        if t == "club":
            call = "Club exercises" if diff > 0 else "Club declines"
        else:
            call = "Declined (the two sides' interests conflict)"
    elif t == "player":
        diff = pred - amt
        reason = f"projected ${pred:.1f}M a year vs option salary ${amt:.1f}M"
        call = "Player declines (market pays more)" if diff > 0 else "Player exercises"
    elif t == "opt-out":
        yrs = r["optout_years"]
        stay = amt / yrs
        leave = pred + buy / yrs
        diff = leave - stay
        reason = f"leaving is worth about ${leave:.1f}M a year vs ${stay:.1f}M a year for staying"
        call = "Opts out" if diff > 0 else "Stays"
    else:
        return np.nan, "Unknown type: needs a manual look", ""
    b = band(diff)
    return diff, ("Toss-up" if b == "toss-up" else f"{call} - {b}"), reason

res = o.apply(decide, axis=1, result_type="expand")
o["gap_m"], o["call"], o["reason"] = res[0], res[1], res[2]

override = {
    "Aroldis Chapman": "Option guaranteed (innings condition met per MLBTR); under contract unless he fails the physical",
    "Gary Sanchez": "Mutual option, terms not listed: likely declined",
}
prefix = {
    "Kyle Freeland": "CHECK FIRST (needs 170 IP in 2026 to vest): ",
    "Nick Pivetta": "VERIFY OPTION TYPE: ",
}
o["call"] = o.apply(
    lambda r: override.get(r["player_name"], prefix.get(r["player_name"], "") + r["call"]), axis=1)

cols = ["player_name", "position", "age_at_signing", "option_type", "amount_m", "buyout_m",
        "pred_aav_m", "gap_m", "call", "reason", "option_note"]
o = o.sort_values("pred_aav_m", ascending=False)[cols].round(1)
o.to_csv("../outputs/fa_2027_option_calls.csv", index=False)

print(o["option_type"].value_counts().to_dict())
print()
print(o["call"].value_counts().to_string())
o[["player_name", "option_type", "pred_aav_m", "call", "reason"]].head(25)

{'club': 25, 'mutual': 25, 'opt-out': 8, 'player': 4}

call
Toss-up                                                                                             34
Club exercises - lean                                                                                6
Declined (the two sides' interests conflict) - lean                                                  5
Club declines - lean                                                                                 4
Stays - strong                                                                                       3
Stays - lean                                                                                         2
Declined (the two sides' interests conflict) - strong                                                2
Club exercises - strong                                                                              1
VERIFY OPTION TYPE: Toss-up                                                                          1
Option guaran

,player_name,option_type,pred_aav_m,call,reason
0,Drew Rasmussen,club,22.9,Toss-up,projected $22.9M a year vs net option cost $19.5M
1,Ronald Acuña Jr.,club,20.8,Club exercises - strong,projected $20.8M a year vs net option cost $7.0M
2,Michael King,opt-out,19.1,Stays - lean,leaving is worth about $21.6M a year vs $29.0M...
5,Nick Pivetta,club,18.0,VERIFY OPTION TYPE: Toss-up,projected $18.0M a year vs net option cost $14.0M
6,Nick Martinez,mutual,17.4,Toss-up,projected $17.4M a year vs net option cost $16.0M
8,Sonny Gray,mutual,16.9,Toss-up,projected $16.9M a year vs net option cost $20.0M
9,Aroldis Chapman,mutual,16.0,Option guaranteed (innings condition met per M...,projected $16.0M a year vs net option cost $12.7M
10,Corbin Burnes,opt-out,15.7,Stays - strong,leaving is worth about $15.7M a year vs $35.0M...
3,Andres Muñoz,club,15.2,Club exercises - lean,projected $15.2M a year vs net option cost $8.0M
12,Bo Bichette,opt-out,14.2,Stays - strong,leaving is worth about $16.7M a year vs $42.0M...


In [10]:
import numpy as np
import pandas as pd

o = pd.read_csv("../data/processed/fa_2027_options.csv")
p = pd.read_csv("../outputs/fa_2027_contract_projections.csv")
o = o.merge(p[["player_name", "position", "age_at_signing", "option_note"]],
            on="player_name", how="left")
o = o[o["player_name"] != "Kyle Freeland"].copy()   # option did not vest (under 170 IP in 2026)

def gap_and_reason(r):
    t, amt, pred = r["option_type"], r["amount_m"], r["pred_aav_m"]
    buy = 0.0 if pd.isna(r["buyout_m"]) else r["buyout_m"]
    if pd.isna(amt):
        return np.nan, "terms missing"
    if t in ("club", "mutual"):
        net = amt - buy
        return pred - net, f"projected ${pred:.1f}M a year vs net option cost ${net:.1f}M"
    if t == "player":
        return pred - amt, f"projected ${pred:.1f}M a year vs option salary ${amt:.1f}M"
    if t == "opt-out":
        yrs = r["optout_years"]
        stay = amt / yrs
        leave = pred + buy / yrs
        return leave - stay, f"leaving is worth about ${leave:.1f}M a year vs ${stay:.1f}M a year for staying"
    return np.nan, "unknown type"

def call_text(t, diff, margin):
    if pd.isna(diff):
        return "Needs a manual look"
    a = abs(diff)
    if a <= margin:
        return "Toss-up"
    strength = "lean" if a <= 2 * margin else "strong"
    if t == "club":
        return f"Club {'exercises' if diff > 0 else 'declines'} - {strength}"
    if t == "player":
        return f"Player {'declines (market pays more)' if diff > 0 else 'exercises'} - {strength}"
    if t == "opt-out":
        return f"{'Opts out' if diff > 0 else 'Stays'} - {strength}"
    if t == "mutual":
        return ("Declined: club would keep him, player tests the market" if diff > 0
                else "Declined: player would take it, club would not")
    return "Needs a manual look"

MARGIN = 4.0   # about the model's average AAV miss, in $M
gr = o.apply(gap_and_reason, axis=1, result_type="expand")
o["gap_m"], o["reason"] = gr[0], gr[1]
o["call"] = [call_text(t, g, MARGIN) for t, g in zip(o["option_type"], o["gap_m"])]

override = {
    "Aroldis Chapman": "Option guaranteed (innings condition met per MLBTR); under contract unless he fails the physical",
    "Gary Sanchez": "Mutual option, terms not listed: likely declined",
}
prefix = {"Nick Pivetta": "VERIFY OPTION TYPE: "}
o["call"] = o.apply(
    lambda r: override.get(r["player_name"], prefix.get(r["player_name"], "") + r["call"]), axis=1)

cols = ["player_name", "position", "age_at_signing", "option_type", "amount_m", "buyout_m",
        "pred_aav_m", "gap_m", "call", "reason", "option_note"]
o = o.sort_values("pred_aav_m", ascending=False)[cols].round(1)
o.to_csv("../outputs/fa_2027_option_calls.csv", index=False)

print("Players:", len(o), "|", o["option_type"].value_counts().to_dict())
print("\nToss-ups at different margins:")
for mg in [2.5, 4.0, 6.0]:
    n = sum(call_text(t, g, mg) == "Toss-up" for t, g in zip(o["option_type"], o["gap_m"]))
    print(f"  margin ${mg}M: {n} of {len(o)}")
print()
print(o["call"].value_counts().to_string())
o[["player_name", "option_type", "pred_aav_m", "call", "reason"]].head(25)

Players: 61 | {'club': 25, 'mutual': 25, 'opt-out': 8, 'player': 3}

Toss-ups at different margins:
  margin $2.5M: 25 of 61
  margin $4.0M: 36 of 61
  margin $6.0M: 47 of 61

call
Toss-up                                                                                             34
Club exercises - lean                                                                                6
Declined: player would take it, club would not                                                       4
Club declines - lean                                                                                 4
Stays - strong                                                                                       3
Declined: club would keep him, player tests the market                                               3
Stays - lean                                                                                         2
Club exercises - strong                                                                           

,player_name,option_type,pred_aav_m,call,reason
0,Drew Rasmussen,club,22.9,Toss-up,projected $22.9M a year vs net option cost $19.5M
1,Ronald Acuña Jr.,club,20.8,Club exercises - strong,projected $20.8M a year vs net option cost $7.0M
2,Michael King,opt-out,19.1,Stays - lean,leaving is worth about $21.6M a year vs $29.0M...
5,Nick Pivetta,club,18.0,VERIFY OPTION TYPE: Toss-up,projected $18.0M a year vs net option cost $14.0M
6,Nick Martinez,mutual,17.4,Toss-up,projected $17.4M a year vs net option cost $16.0M
8,Sonny Gray,mutual,16.9,Toss-up,projected $16.9M a year vs net option cost $20.0M
9,Aroldis Chapman,mutual,16.0,Option guaranteed (innings condition met per M...,projected $16.0M a year vs net option cost $12.7M
10,Corbin Burnes,opt-out,15.7,Stays - strong,leaving is worth about $15.7M a year vs $35.0M...
3,Andres Muñoz,club,15.2,Club exercises - lean,projected $15.2M a year vs net option cost $8.0M
12,Bo Bichette,opt-out,14.2,Stays - strong,leaving is worth about $16.7M a year vs $42.0M...


In [11]:
import pandas as pd

o = pd.read_csv("../outputs/fa_2027_option_calls.csv")
o["call_group"] = (o["call"]
    .str.replace(r" - (lean|strong)$", "", regex=True)
    .str.replace(r"^Declined: club.*", "Declined (club would keep)", regex=True)
    .str.replace(r"^Declined: player.*", "Declined (player would take)", regex=True)
    .str.replace(r"^Option guaranteed.*", "Guaranteed", regex=True)
    .str.replace(r"^Mutual option, terms.*", "No terms listed", regex=True)
    .str.replace(r"^VERIFY OPTION TYPE: ", "", regex=True))
print(pd.crosstab(o["option_type"], o["call_group"]).to_string())

call_group   Club declines  Club exercises  Declined (club would keep)  Declined (player would take)  Guaranteed  No terms listed  Player exercises  Stays  Toss-up
option_type                                                                                                                                                        
club                     4               7                           0                             0           0                0                 0      0       14
mutual                   0               0                           3                             4           1                1                 0      0       16
opt-out                  0               0                           0                             0           0                0                 0      5        3
player                   0               0                           0                             0           0                0                 1      0        2


In [12]:
import pandas as pd, unicodedata

def nm(s):
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode().lower()
    return s.replace(".", "").replace("-", " ").replace(" jr", "").strip()

kept = ["Ronald Acuña Jr.", "Ozzie Albies", "Andres Muñoz", "Sandy Alcantara",
        "Drew Rasmussen", "Bo Bichette", "Corbin Burnes", "Ryan Helsley", "Tatsuya Imai"]
left = ["Michael King", "Clay Holmes", "Dustin May", "Matthew Boyd", "Carlos Estevez",
        "Jeff McNeil", "Victor Robles", "Jeffrey Springs", "Trevor Stephan",
        "Nick Martinez", "Cedric Mullins", "Zach Eflin", "Nick Pivetta",
        "Luis Robert Jr.", "Sonny Gray"]
actual = {nm(n): "kept" for n in kept} | {nm(n): "left" for n in left}

o = pd.read_csv("../outputs/fa_2027_option_calls.csv")
o["actual"] = o["player_name"].map(lambda n: actual.get(nm(n)))

def predicted(call):
    if call.startswith(("Club exercises", "Stays", "Player exercises")): return "kept"
    if call.startswith(("Club declines", "Declined", "Opts out")): return "left"
    return None

o["predicted"] = o["call"].map(predicted)
done = o[o["actual"].notna()]
clear = done[done["predicted"].notna()]
print("decided:", len(done), "| clear calls:", len(clear),
      "| correct:", (clear["predicted"] == clear["actual"]).sum())
print("toss-ups that resolved:", done["predicted"].isna().sum())
print(clear[clear["predicted"] != clear["actual"]][["player_name", "call", "actual"]])

decided: 23 | clear calls: 11 | correct: 9
toss-ups that resolved: 12
        player_name                  call actual
2      Michael King          Stays - lean   left
12  Sandy Alcantara  Club declines - lean   kept
